In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from sklearn.metrics import f1_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VIDEO_DIR = "./datasets/data-from-juniors/videos"
CSV_PATH = "./datasets/data-from-juniors/dataset.csv"

NUM_FRAMES = 8
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 5
df = pd.read_csv(CSV_PATH)

rare_labels = [
    "ethinity_hate",
    "caste_based_hate",
    "social_hate",
    "religion_hate",
    "controversial"
]

df["rare_hate"] = df[rare_labels].max(axis=1)
df = df.drop(columns=rare_labels)


LABEL_COLUMNS = [
    'sensitive','derogatory__lang','threat','sexuality_hate',
    'nationality_hate','political_hate','anger','emotional',
    'indv_hate','gender_hate','rare_hate'
]

# keep only needed columns
df = df[["video_id"] + LABEL_COLUMNS]

# remove empty label rows
df["label_sum"] = df[LABEL_COLUMNS].sum(axis=1)
df = df[df["label_sum"] > 0]
df = df.drop(columns=["label_sum"])

df = df.reset_index(drop=True)

print("Final samples:", len(df))
df
label_counts = df[LABEL_COLUMNS].sum()
total = len(df)

pos_weight = (total - label_counts) / label_counts
pos_weight = pos_weight.clip(upper=20)

pos_weight = torch.tensor(pos_weight.values, dtype=torch.float).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
class FrameDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        clip_id = os.path.splitext(row["video_id"])[0]
        frame_dir = os.path.join("./datasets/data-from-juniors/frames", clip_id)

        frame_files = sorted(os.listdir(frame_dir))[:8]

        images = []
        for f in frame_files:
            img = cv2.imread(os.path.join(frame_dir, f))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)

        inputs = self.processor(images=images, return_tensors="pt")

        labels = torch.tensor(row[LABEL_COLUMNS].values.astype(float))

        return {
            "pixel_values": inputs["pixel_values"],
            "labels": labels
        }
def collate_fn(batch):
    pixel_values = torch.stack([b["pixel_values"] for b in batch])
    labels = torch.stack([b["labels"] for b in batch])
    return pixel_values, labels
dataset = FrameDataset(df)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn
)
class VisionModel(nn.Module):
    def __init__(self, num_labels):
        super().__init__()

        self.clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        hidden = self.clip.config.projection_dim

        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, num_labels)
        )

    def forward(self, pixel_values):
        B, F, C, H, W = pixel_values.shape

        pixel_values = pixel_values.view(B * F, C, H, W)

        features = self.clip.get_image_features(pixel_values=pixel_values)

        # normalize (IMPORTANT)
        features = features / features.norm(dim=-1, keepdim=True)

        features = features.view(B, F, -1)
        features = features.mean(dim=1)

        logits = self.classifier(features)
        return logits
model = VisionModel(len(LABEL_COLUMNS)).to(DEVICE)

# freeze backbone first
for param in model.clip.parameters():
    param.requires_grad = False

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
import logging
import time
import torch

def train_epoch(epoch):
    model.train()

    total_loss = 0
    start_time = time.time()

    # accuracy trackers
    all_preds = []
    all_labels = []

    logging.info(f"Epoch {epoch} started")

    for step, (pixel_values, labels) in enumerate(train_loader):
        batch_start = time.time()

        pixel_values = pixel_values.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(pixel_values)
        probs = torch.sigmoid(logits)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ---- STORE PREDICTIONS ----
        preds = (probs > 0.5).detach().cpu()
        all_preds.append(preds)
        all_labels.append(labels.detach().cpu())

        # ---- LOG EVERY 20 STEPS ----
        if step % 20 == 0:
            elapsed = time.time() - batch_start

            logging.info(
                f"[Epoch {epoch} | Step {step}/{len(train_loader)}] "
                f"Loss: {loss.item():.4f} | "
                f"Batch Time: {elapsed:.2f}s"
            )

        # ---- DEBUG: NAN CHECK ----
        if torch.isnan(loss):
            logging.error(f"NaN loss detected at step {step}")
            break

    # ---- AGGREGATE METRICS ----
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    # Hamming accuracy (recommended)
    hamming_acc = (all_preds == all_labels).mean()

    # Exact match accuracy (strict)
    exact_match_acc = (all_preds == all_labels).all(axis=1).mean()

    avg_loss = total_loss / len(train_loader)
    epoch_time = time.time() - start_time

    logging.info(
        f"Epoch {epoch} completed | "
        f"Avg Loss: {avg_loss:.4f} | "
        f"Hamming Acc: {hamming_acc:.4f} | "
        f"Exact Acc: {exact_match_acc:.4f} | "
        f"Time: {epoch_time:.2f}s"
    )

    return {
        "loss": avg_loss,
        "hamming_acc": hamming_acc,
        "exact_acc": exact_match_acc
    }
import logging
import time
import numpy as np
from sklearn.metrics import f1_score

def evaluate(epoch, loader):
    model.eval()

    all_preds = []
    all_labels = []

    total_loss = 0
    total_samples = 0

    start_time = time.time()

    logging.info(f"========== EVAL START | Epoch {epoch} ==========")

    with torch.no_grad():
        for step, (pixel_values, labels) in enumerate(loader):
            batch_start = time.time()

            pixel_values = pixel_values.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(pixel_values)
            probs = torch.sigmoid(logits)

            loss = criterion(logits, labels)

            total_loss += loss.item() * pixel_values.size(0)
            total_samples += pixel_values.size(0)

            preds = (probs > 0.5).cpu().numpy()

            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

            # ---- STEP LOG ----
            if step % 20 == 0:
                elapsed = time.time() - batch_start
                logging.info(
                    f"[Eval | Epoch {epoch} | Step {step}/{len(loader)}] "
                    f"Loss: {loss.item():.4f} | "
                    f"Batch Time: {elapsed:.2f}s"
                )

    # ---- AGGREGATE ----
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    val_loss = total_loss / total_samples

    # ---- METRICS ----
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average="micro", zero_division=0)
    per_label_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)

    # ---- ACCURACY (MULTI-LABEL SAFE) ----
    exact_match_acc = (all_preds == all_labels).all(axis=1).mean()
    hamming_acc = (all_preds == all_labels).mean()

    elapsed = time.time() - start_time

    # ---- LOG RESULTS ----
    logging.info("========== EVAL RESULTS ==========")
    logging.info(f"Epoch {epoch} | Val Loss: {val_loss:.4f}")
    logging.info(f"Macro F1: {macro_f1:.4f}")
    logging.info(f"Micro F1: {micro_f1:.4f}")
    logging.info(f"Hamming Accuracy: {hamming_acc:.4f}")
    logging.info(f"Exact Match Accuracy: {exact_match_acc:.4f}")
    logging.info(f"Eval Time: {elapsed:.2f}s")

    logging.info("---------- Per Label F1 ----------")
    for i, label in enumerate(LABEL_COLUMNS):
        logging.info(f"{label}: {per_label_f1[i]:.4f}")

    logging.info("=================================")

    return {
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "hamming_acc": hamming_acc,
        "exact_match_acc": exact_match_acc,
        "per_label_f1": per_label_f1
    }
best_f1 = 0

for epoch in range(EPOCHS):
    # ---- TRAIN ----
    train_metrics = train_epoch(epoch)

    # ---- EVAL ----
    val_metrics = evaluate(epoch, val_loader)

    train_loss = train_metrics["loss"]
    train_acc = train_metrics["hamming_acc"]

    val_loss = val_metrics["val_loss"]
    val_f1 = val_metrics["macro_f1"]

    # ---- LOG ----
    print(
        f"Epoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val F1: {val_f1:.4f}"
    )

    # ---- UNFREEZE BACKBONE ----
    if epoch == 2:
        print("Unfreezing CLIP backbone...")
        for param in model.clip.parameters():
            param.requires_grad = True

    # ---- SAVE BEST MODEL ----
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_f1": best_f1
        }, "vision_model.pt")

        print(f"Saved best model | F1: {best_f1:.4f}")

KeyboardInterrupt: 